In [1]:
from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np

BASE = Path(r'C:\Users\suuser\Desktop\minlplib')
FINAL = BASE / 'Areview' / 'combined_minlplib_qplib'
# You may override any source explicitly if autodiscovery has multiple candidates.
ROOT_IMPORTANCE_CSV = None
BB_IMPORTANCE_CSV = None
ABLATION_CSV = None
OUTPUT = FINAL / 'feature_interpretation_summaries'
OUTPUT.mkdir(parents=True, exist_ok=True)

def discover(name, override=None):
    if override:
        p = Path(override)
        if not p.exists(): raise FileNotFoundError(p)
        return p
    matches = sorted(FINAL.rglob(name), key=lambda p: p.stat().st_mtime, reverse=True)
    if not matches: raise FileNotFoundError(f'Missing {name} under {FINAL}. Run the producing notebook first.')
    print(f'{name}: {matches[0]}')
    if len(matches)>1:
        print('  Other available runs:', *[str(p) for p in matches[1:5]], sep='\n    ')
        print('  WARNING: verify that the chosen file belongs to the final experiment.')
    return matches[0]

root_file = discover('all_root_feature_importance.csv', ROOT_IMPORTANCE_CSV)
bb_file = discover('all_feature_importance.csv', BB_IMPORTANCE_CSV)
abl_file = discover('all_ablation_results.csv', ABLATION_CSV)
root = pd.read_csv(root_file); bb = pd.read_csv(bb_file); abl = pd.read_csv(abl_file)
print('Loaded:', len(root), 'root importance rows;',len(bb),'B&B/mixed importance rows;',len(abl),'ablation rows')
print('Root columns:',list(root.columns));print('Other importance columns:',list(bb.columns));print('Ablation columns:',list(abl.columns))

all_root_feature_importance.csv: C:\Users\suuser\Desktop\minlplib\Areview\combined_minlplib_qplib\root_node_feature_importance_20261024_172152_664157\all_root_feature_importance.csv
all_feature_importance.csv: C:\Users\suuser\Desktop\minlplib\Areview\combined_minlplib_qplib\feature_importance_6_static_20261022_004158_380784\all_feature_importance.csv
all_ablation_results.csv: C:\Users\suuser\Desktop\minlplib\Areview\combined_minlplib_qplib\ablation_12_models_extended_20261022_020524_290117\all_ablation_results.csv
  Other available runs:
    C:\Users\suuser\Desktop\minlplib\Areview\combined_minlplib_qplib\ablation_12_models_extended_20261022_020012_503832\all_ablation_results.csv
    C:\Users\suuser\Desktop\minlplib\Areview\combined_minlplib_qplib\ablation_12_models_extended_20261022_014446_428262\all_ablation_results.csv
    C:\Users\suuser\Desktop\minlplib\Areview\combined_minlplib_qplib\ablation_6_static_ID_FIXED_20261022_012429_321955\all_ablation_results.csv
Loaded: 512 root impor

In [2]:
# Normalise importance outputs without assuming model/experiment labels match perfectly.
def normalize_importance(df, source):
    df=df.copy()
    cols={c.lower():c for c in df.columns}
    def first(*names):
        for nm in names:
            if nm.lower() in cols: return cols[nm.lower()]
        return None
    ft=first('feature'); value=first('importance_mean','permutation_importance_mean')
    if ft is None or value is None: raise ValueError(f'{source}: missing feature/importance column')
    e=first('experiment','design_id'); model=first('model','selected_model'); method=first('method','comparison'); training=first('train_setting','training_setting')
    out=pd.DataFrame({'source':source,'feature':df[ft].astype(str),'importance':pd.to_numeric(df[value],errors='coerce')})
    out['experiment']=df[e].astype(str) if e else source
    out['model']=df[model].astype(str) if model else 'not_reported'
    out['setting']=df[training].astype(str) if training else ''
    out['method']=df[method].astype(str) if method else ''
    out['source_row']=np.arange(len(df))
    return out.dropna(subset=['importance'])

ri=normalize_importance(root,'root'); bi=normalize_importance(bb,'bb_existing')
# Existing B&B importance output may ALREADY contain root entries: avoid duplicates.
if bi['experiment'].str.contains('root',case=False).any():
    print('NOTE: B&B importance CSV also contains root experiments; retaining them as separately sourced rows for audit.')
all_imp=pd.concat([ri,bi],ignore_index=True)
all_imp.to_csv(OUTPUT/'feature_importance_all_sources.csv',index=False)
# Feature rankings per experiment/model; negative importances not treated as positive evidence.
top=(all_imp.sort_values('importance',ascending=False)
     .groupby(['source','experiment','model'],dropna=False).head(15))
top.to_csv(OUTPUT/'feature_importance_top15_by_experiment.csv',index=False)
# Across-model recurrence: number of distinct contexts in which a feature is in top 15.
rec=(top.groupby('feature').agg(n_contexts=('experiment','nunique'),median_importance=('importance','median'),max_importance=('importance','max')).reset_index()
     .sort_values(['n_contexts','median_importance'],ascending=False))
rec.to_csv(OUTPUT/'feature_importance_recurrence.csv',index=False)
display(rec.head(20))

NOTE: B&B importance CSV also contains root experiments; retaining them as separately sourced rows for audit.


,feature,n_contexts,median_importance,max_importance
3,PH_A0,13,0.067593,0.213451
50,lambda_min_A0_scaled,13,0.018211,0.092891
51,rho_A0_over_n,11,0.026384,0.105010
49,lambda_max_A0_scaled,11,0.023148,0.121264
7,eta_A0_over_n,9,0.043367,0.170341
52,theta_A0,9,0.012131,0.151902
38,h_max_Rho,8,0.006964,0.024205
31,h_avg_Lambda_min_scaled,8,0.006496,0.037695
40,h_max_Xi,8,0.004304,0.022696
35,h_avg_Xi,8,0.002992,0.016865


In [3]:
# Ablation is BOTH root and B&B: compute meaningful drops from full-feature baseline.
required={'experiment','model','task','variant'}
if not required.issubset(abl): raise ValueError(f'Ablation file missing columns: {required-set(abl.columns)}')
metric_columns={'classification': 'balanced_accuracy', 'regression':'rAccuracy_tol'}
rows=[]
for (exp,model,task),grp in abl.groupby(['experiment','model','task']):
    metric=metric_columns.get(task)
    if metric not in grp or 'Full_64' not in set(grp.variant): continue
    base=pd.to_numeric(grp.loc[grp.variant=='Full_64',metric],errors='coerce').dropna()
    if len(base)!=1: continue
    for _,r in grp.iterrows():
        if r['variant']=='Full_64':continue
        score=pd.to_numeric(pd.Series([r[metric]]),errors='coerce').iloc[0]
        if pd.isna(score):continue
        rows.append({'experiment':exp,'method':r.get('method',''),'training_setting':r.get('training_setting',''),
                     'model':model,'task':task,'variant':r['variant'],'metric':metric,
                     'full_score':float(base.iloc[0]),'ablated_score':float(score),
                     'drop':float(base.iloc[0]-score)})
drops=pd.DataFrame(rows)
if drops.empty: print('No compatible ablation comparisons found. Check variants and metric columns.')
else:
    drops.to_csv(OUTPUT/'ablation_all_drops.csv',index=False)
    meaningful=drops[drops['drop']>0].sort_values('drop',ascending=False)
    meaningful.to_csv(OUTPUT/'ablation_positive_drops.csv',index=False)
    display(meaningful.groupby(['method','task','variant'],dropna=False).agg(
        mean_drop=('drop','mean'),n_experiments=('experiment','nunique')).sort_values('mean_drop',ascending=False).head(25))
print('Outputs saved in:',OUTPUT)
print('Interpretation caution: compare feature importance within each experiment only; metric scales differ. Ablation negative drops mean removal improved performance.')

mean_drop  n_experiments
method task           variant                                             
root   classification Only_objective_and_bounds    0.415355              2
bb     classification No_BoundsExist               0.239662              1
       regression     Only_constraint_and_bounds   0.171429              1
                      No_objective_A0              0.171429              1
                      Only_objective_and_bounds    0.164286              1
root   classification No_objective_shape           0.111420              3
bb     regression     No_h_central                 0.110714              1
root   classification No_h_central                 0.110302              2
                      No_constraint_h              0.110229              3
                      No_objective_or_h            0.109942              4
       regression     No_objective_or_h            0.101942              2
bb     regression     No_constraint_h              0.100000              1
       classification No_objective_shape           0.095494              2
root   regression     No_objective_A0              0.092233              2
bb     classification No_constraint_h              0.086748              2
                      No_h_central                 0.077080              2
root   classification No_BoundsExist               0.075911              2
bb     classification No_objective_or_h            0.071879              2
root   regression     Only_constraint_and_bounds   0.071348              3
bb     classification Only_objective_and_bounds    0.069925              2
root   regression     No_objective_extremes        0.058252              2
bb     regression     No_structure_g_comm          0.050000              1
root   classification Only_constraint_and_bounds   0.047148              4
bb     classification Only_constraint_and_bounds   0.044501              2
root   classification No_objective_A0              0.044277              4

Outputs saved in: C:\Users\suuser\Desktop\minlplib\Areview\combined_minlplib_qplib\feature_interpretation_summaries
Interpretation caution: compare feature importance within each experiment only; metric scales differ. Ablation negative drops mean removal improved performance.
